In [ ]:
# Colab setup -- installs SoftMobility when running on Google Colab.
# Safe to run locally: it does nothing outside Colab.
try:
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/C0PEP0D/SoftMobility.git
except ImportError:
    pass

# Optimization of a three-sphere swimmer

### Imports

In [ ]:
import jax.numpy as jnp
import numpy as np
import optax

import softmobility as sm
from softmobility.tutorials import figstyle

figstyle.apply()
np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Parameters

In [2]:
# Create a quiet flow
no_flow = sm.no_flow()

In [3]:
# numerical simulation parameters
FINAL_TIME = 24 * jnp.pi    # final time of simulations
N_DT = 64                   # number of time steps per period

# Optimization parameters
N_TRANSIENT = 4            # number of periods before optimization begins

# Calculated paramters
DT = 2.0 * np.pi / N_DT
N_STEPS = int(FINAL_TIME / DT)

### YAML file

In [4]:
yaml_data = """
# Variable Prefixes (Optional)
design_names:
  - spring_k
  - radius
  - distance

# Default Values (Optional)
defaults:
  spring_k: .1
  distance0: 0.2
  radius1: 0.1
  radius2: 0.1
  _spr_: 100
  _rad_: 1
  _len_: 10
  _act_: 1

# Spheres (Mandatory)
spheres:
  - # Sphere center #################
    radius: 0.1
    force: [-_spr_ * spring_k * dof0, 0, 0]

  - # Sphere left ###################
    radius: _rad_ * radius1
    position: [-(_len_ * distance0 + dof0) , 0, 0]
    force: [_spr_ * spring_k * dof0, 0, 0]

  - # Sphere right ##################
    radius: _rad_ * radius2
    position: [_act_ * (2 + sin(time)), 0, 0]
"""

## Simulation of default parameters

### Soft Body

In [5]:
mybody= sm.SoftBody(yaml_data, verbose=True)
print("\nDesign variables\n", mybody.design_variables)
print(mybody.design_defaults)

  Found variables: dof0, distance0, radius1, radius2, spring_k, 
    Sphere 0
      Radius: 0.100000000000000
      Position: ['0', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[['-100*spring_k'], ['0'], ['0'], ['0'], ['0'], ['0']]
    Sphere 1
      Radius: radius1
      Position: ['-10*distance0 - dof0', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[['100*spring_k'], ['0'], ['0'], ['0'], ['0'], ['0']]
    Sphere 2
      Radius: radius2
      Position: ['sin(time) + 2', '0', '0']
      Orientation: ['0', '0', '0']
      Coupling matrix C_H:
[[], [], [], [], [], []]
      Coupling matrix C_K:
[['0'], ['0'], ['0'], ['0'], ['0'], ['0']]

Design variables
 ['distance0', 'radius1', 'radius2', 'spring_k']
[ 0.2  0.1  0.1  0.1]


### Flow-body solver

In [6]:
ROLLOUT = sm.FlowBodyRollout(
    soft_body=mybody,
    flow=no_flow,
)

In [7]:
positions, orientations, dofs = ROLLOUT.rollout(DT, N_STEPS)

### Function for figure 

In [ ]:
def make_figure(positions, orientations, dofs):
    time = jnp.linspace(0, FINAL_TIME, N_STEPS)

    fig, axes = figstyle.subplots(size="full", aspect=658 / 540, nrows=3, sharex=True)

    for i in range(mybody.Ndof):
        axes[0].plot(time, dofs[:, i], linewidth=1.2, label=f"DOF {i}")
    axes[0].set_ylabel("DOF")
    axes[0].set_title("Trajectory data over time")
    axes[0].legend(frameon=False, loc="upper right")

    for j, name in enumerate(("Position X", "Position Y", "Position Z")):
        axes[1].plot(time, positions[:, j], linewidth=1.2, label=name)
    axes[1].set_ylabel("Position")
    axes[1].legend(frameon=False, loc="upper right")

    for j, name in enumerate(("Orientation X", "Orientation Y", "Orientation Z")):
        axes[2].plot(time, orientations[:, j], linewidth=1.2, label=name)
    axes[2].set_xlabel("time")
    axes[2].set_ylabel("Orientation")
    axes[2].legend(frameon=False, loc="upper right")

    return fig

### Figure

In [ ]:
make_figure(positions, orientations, dofs)

## Numerical optimization

### Objective

In [10]:
def myobjective(myrollout, design):
    # Run N_TRANSIENT periods
    positions, orientations, dofs = myrollout.rollout(
        dt=DT,
        n_steps=N_DT * N_TRANSIENT,
        design=design)

    # Run one period for evaluation of mean velocity along x
    x_init = positions[-1,0]   # capture before evaluation
    positions, orientations, dofs = myrollout.rollout(
        dt=DT,
        n_steps=N_DT,
        init_position=positions[-1],
        init_orientation=orientations[-1],
        init_dofs=dofs[-1],
        design=design)

    return (positions[-1,0] - x_init) / (2. * np.pi)

### Optimization

In [11]:
myoptimizer = sm.FlowBodyOptimizer(ROLLOUT, myobjective, optimizer=optax.adam(0.001))

opt_design = myoptimizer.run(
    init_design = mybody.design_defaults,
    n_steps     = 1001,
    print_every = 100,
    clip_min    = 0.01,
    clip_max    = 1.0,
    grad_clip   = 0.1,
    maximize    = False,    # True for x-component of velocity > 0
)

print("\nOPTIMUM DESIGN PARAMETERS:\n", mybody.design_variables)
print(opt_design)

step    0  objective=0.0005  |grad|=0.0096  design0=0.20  design1=0.10  design2=0.10  design3=0.10
step  100  objective=0.0157  |grad|=0.0982  design0=0.05  design1=0.16  design2=0.22  design3=0.03
step  200  objective=0.0196  |grad|=0.0012  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step  300  objective=0.0196  |grad|=0.0000  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step  400  objective=0.0196  |grad|=0.0000  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step  500  objective=0.0196  |grad|=0.0000  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step  600  objective=0.0196  |grad|=0.0000  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step  700  objective=0.0196  |grad|=0.0000  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step  800  objective=0.0196  |grad|=0.0000  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step  900  objective=0.0196  |grad|=0.0000  design0=0.03  design1=0.11  design2=0.21  design3=0.03
step 1000 

### Figure

In [ ]:
positions, orientations, dofs = ROLLOUT.rollout(DT, N_STEPS, design=opt_design)
make_figure(positions, orientations, dofs)

## Snapshot strip

Static snapshots of the swimmer at evenly spaced phases over a single
actuation period. This replaces the plotly slider/animation: matplotlib
has no clean inline-animation equivalent and a strip of snapshots is
sufficient to see the swimmer's stroke.

### Function

In [ ]:
import matplotlib.pyplot as plt

from softmobility.classes.solver import rotation_matrix


def plot_snapshots(softbody, positions, orientations, dofs, dt,
                   design=None, n_snapshots=6, sphere_alpha=0.55):
    """Render n_snapshots evenly spaced lab-frame poses of the swimmer
    as a horizontal strip of 3D panels."""
    if design is None:
        design = softbody.design_defaults

    N = positions.shape[0]
    idx = np.linspace(0, N - 1, n_snapshots).astype(int)

    # Pre-compute per-frame sphere lab positions/radii to size the cube.
    poses = []
    for i in idx:
        pos_i = jnp.asarray(positions[i])
        R_body = rotation_matrix(jnp.asarray(orientations[i]))
        dof_i = jnp.asarray(dofs[i])
        time = jnp.asarray([i * dt])
        spheres_lab = []
        for sphere in softbody.spheres:
            s_pos = np.array(sphere.position(dof_i, design, time))
            s_rad = float(sphere.radius(dof_i, design))
            spheres_lab.append((np.asarray(pos_i) + np.asarray(R_body) @ s_pos, s_rad))
        poses.append(spheres_lab)

    all_lo = np.array([
        [c[0] - r, c[1] - r, c[2] - r]
        for snap in poses for (c, r) in snap
    ]).min(axis=0)
    all_hi = np.array([
        [c[0] + r, c[1] + r, c[2] + r]
        for snap in poses for (c, r) in snap
    ]).max(axis=0)
    pad = 0.1 * (all_hi - all_lo).max()
    bound_lo, bound_hi = all_lo.min() - pad, all_hi.max() + pad

    w = figstyle.SIZES["full"]
    fig = plt.figure(figsize=(w, w / n_snapshots))
    for k, (frame_idx, snap) in enumerate(zip(idx, poses, strict=True)):
        ax = fig.add_subplot(1, n_snapshots, k + 1, projection="3d")
        ax.set_proj_type("ortho")
        ax.view_init(elev=28.8, azim=45)
        ax.set_box_aspect((1, 1, 1))
        ax.set_axis_off()
        for centre, radius in snap:
            figstyle.add_sphere(ax, centre, radius,
                                color=figstyle.COLORS["blue"],
                                alpha=sphere_alpha,
                                contour_color=figstyle.COLORS["blue"],
                                contour_width=1.0)
        # Body-center trajectory up to this snapshot.
        ax.plot(positions[: frame_idx + 1, 0],
                positions[: frame_idx + 1, 1],
                positions[: frame_idx + 1, 2],
                color=figstyle.COLORS["grey"], linewidth=1.0)
        ax.set_xlim(bound_lo, bound_hi)
        ax.set_ylim(bound_lo, bound_hi)
        ax.set_zlim(bound_lo, bound_hi)
        ax.set_title(f"$t = {frame_idx * dt:.1f}$", fontsize=9)

    fig.tight_layout()
    return fig

### Plot

In [ ]:
plot_snapshots(
    mybody, positions, orientations, dofs,
    design=opt_design, dt=DT, n_snapshots=6,
)